# 00 — Exploratory Data Analysis: Athletes

**Purpose**: Understand the raw `summerOly_athletes.csv` before writing cleaning code.
Each section checks one aspect of data quality. Findings feed directly into `src/preprocess/clean_athletes.py`.

Reference: PIPELINE.md Section 1.1

## 0. Setup

In [5]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.utils.config import ATHLETES_FILE, DATA_DICT_FILE
from src.utils.io_utils import read_csv

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 20)

print(f'Project root: {project_root}')
print(f'Data file  : {ATHLETES_FILE}')

Project root: d:\Vault\Future\2025_Problem_C
Data file  : D:\Vault\Future\2025_Problem_C\data\summerOly_athletes.csv


## 1. Basic Shape & Structure

**Why**: Confirm we have the expected 252,565 rows and 9 columns. Any deviation means the data file may have been corrupted or truncated.

In [6]:
df = read_csv(ATHLETES_FILE)

print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print()
print('Column list:')
for i, col in enumerate(df.columns):
    print(f'  [{i}] {col}')
print()
print('dtypes:')
print(df.dtypes)

Rows: 252,565
Columns: 9

Column list:
  [0] Name
  [1] Sex
  [2] Team
  [3] NOC
  [4] Year
  [5] City
  [6] Sport
  [7] Event
  [8] Medal

dtypes:
Name       str
Sex        str
Team       str
NOC        str
Year     int64
City       str
Sport      str
Event      str
Medal      str
dtype: object


## 2. Missing Values

**Why**: Missing values in `Medal` are expected (many athletes didn't win). Missing values in `NOC`, `Year`, or `Event` would indicate data corruption. Knowing which columns have NaN helps us design proper fill/drop logic in the cleaning script.

In [7]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})
print(missing_df[missing_df['missing_count'] > 0])

Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


## 3. Sample Rows — Head, Tail, Random

**Why**: Visually inspect the data to spot obvious anomalies (garbled text, weird values, etc.). Random sampling shows rows that head/tail might miss.

In [8]:
print('=== HEAD (first 5) ===')
display(df.head())
print('\n=== TAIL (last 5) ===')
display(df.tail())
print('\n=== RANDOM (5 rows) ===')
display(df.sample(5, random_state=42))

=== HEAD (first 5) ===


,Name,Sex,Team,NOC,Year,City,Sport,Event,Medal
0,Ahmad Abouwi,M,Afghanistan,AFG,1956,Melbourne,Hockey,Hockey Men's Hockey,No medal
1,Jammal-ud-Din Affendi,M,Afghanistan,AFG,1936,Berlin,Hockey,Hockey Men's Hockey,No medal
2,Mohammad Afzal,M,Afghanistan,AFG,1948,London,Football,Football Men's Football,No medal
3,Mohammad Aktar,M,Afghanistan,AFG,1980,Moskva,Wrestling,"Wrestling Men's Light-Flyweight, Freestyle",No medal
4,Mohammad Anwary,M,Afghanistan,AFG,1964,Tokyo,Wrestling,"Wrestling Men's Bantamweight, Freestyle",No medal



=== TAIL (last 5) ===


,Name,Sex,Team,NOC,Year,City,Sport,Event,Medal
252560,Joan Nyahora,F,Zimbabwe,ZIM,2024,Paris,Athletics,Women's Marathon,No medal
252561,Isaac Mpofu,M,Zimbabwe,ZIM,2024,Paris,Athletics,Men's Marathon,No medal
252562,Makanakaishe Charamba,M,Zimbabwe,ZIM,2024,Paris,Athletics,Men's 200m,No medal
252563,van Paige,F,Zimbabwe,ZIM,2024,Paris,Swimming,Women's 100m Freestyle,No medal
252564,Denilson Cyprianos,M,Zimbabwe,ZIM,2024,Paris,Swimming,Men's 200m Backstroke,No medal



=== RANDOM (5 rows) ===


,Name,Sex,Team,NOC,Year,City,Sport,Event,Medal
200029,Lala Wane,F,Senegal,SEN,2016,Rio de Janeiro,Basketball,Basketball Women's Basketball,No medal
211421,Ingrid (-grane),F,Sweden,SWE,1952,Helsinki,Swimming,Swimming Women's 400 metres Freestyle,No medal
107366,Karin Schnaase,F,Germany,GER,2016,Rio de Janeiro,Badminton,Badminton Women's Singles,No medal
251058,Franjo Mihali,M,Yugoslavia,YUG,1952,Helsinki,Athletics,"Athletics Men's 10,000 metres",No medal
196023,Aleksandr Danilov,M,Russia,RUS,1996,Atlanta,Shooting,"Shooting Men's Free Pistol, 50 metres",No medal


## 4. Medal Values — Unique Check

**Why**: PIPELINE.md says valid values are {Gold, Silver, Bronze, No medal}. We need to see all *actual* values in the column to know what normalization is required (e.g. "No Medal" vs "No medal").

In [9]:
medal_vals = df['Medal'].value_counts(dropna=False)
print(f'Unique Medal values: {df["Medal"].nunique()}')
print()
for val, count in medal_vals.items():
    flag = ''
    if pd.isna(val):
        flag = ' <-- NaN'
    elif val not in {'Gold', 'Silver', 'Bronze', 'No medal'}:
        flag = ' <-- NEEDS NORMALIZATION'
    print(f'  [{val}] : {count:,}{flag}')

Unique Medal values: 4

  [No medal] : 213,747
  [Bronze] : 13,070
  [Gold] : 13,002
  [Silver] : 12,746


## 5. Year Distribution

**Why**: Confirm years are in [1896, 2024], every 4 years, and check for any anomalous years (e.g., cancelled years 1916/1940/1944 appearing). Knowing the year span tells us how many Olympiads we have data for.

In [10]:
from src.utils.config import CANCELLED_YEARS, OLYMPIC_YEARS

print(f'Year range: {df["Year"].min()} - {df["Year"].max()}')
print(f'Unique years in data: {sorted(df["Year"].unique())}')
print()

# Check for cancelled years
bad_years = set(df['Year'].unique()) & CANCELLED_YEARS
if bad_years:
    print(f'CANCELLED YEARS FOUND: {bad_years}')
else:
    print('No cancelled years in data')

# Rows per year
print('\nRows per Olympiad:')
display(df['Year'].value_counts().sort_index())

Year range: 1896 - 2024
Unique years in data: [np.int64(1896), np.int64(1900), np.int64(1904), np.int64(1906), np.int64(1908), np.int64(1912), np.int64(1920), np.int64(1924), np.int64(1928), np.int64(1932), np.int64(1936), np.int64(1948), np.int64(1952), np.int64(1956), np.int64(1960), np.int64(1964), np.int64(1968), np.int64(1972), np.int64(1976), np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1996), np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016), np.int64(2020), np.int64(2024)]

No cancelled years in data

Rows per Olympiad:


Year
1896      380
1900     1936
1904     1301
1906     1733
1908     3101
        ...  
2008    13602
2012    12920
2016    13688
2020    15121
2024    14892
Name: count, Length: 31, dtype: int64

## 6. NOC vs Team — The Critical Relationship

**Why**: This is the SINGLE most important check for Phase 1. The same NOC code can have multiple Team name spellings (e.g. 'United States' vs 'United States-1'). We need to discover all (NOC, Team) pairs to build the NOC unification table (Step 1.5).

Questions we answer:
- How many unique NOC codes?
- How many unique Team names?
- Which NOCs have multiple Team spellings?
- Which Team names contain a dash-suffix pattern (multi-team indicator)?

In [11]:
noc_team = df.groupby('NOC')['Team'].apply(lambda x: sorted(x.unique())).reset_index()
noc_team['n_team_names'] = noc_team['Team'].apply(len)

print(f'Unique NOC codes  : {df["NOC"].nunique()}')
print(f'Unique Team names : {df["Team"].nunique()}')
print()

# NOCs with multiple Team spellings - these need the unification table
multi_name = noc_team[noc_team['n_team_names'] > 1].sort_values('n_team_names', ascending=False)
print(f'NOCs with >1 Team spelling: {len(multi_name)}')
print()
print('=== Top 20 NOCs with most Team name variants ===')
for _, row in multi_name.head(20).iterrows():
    print(f'  {row["NOC"]} ({row["n_team_names"]}): {row["Team"]}')

Unique NOC codes  : 234
Unique Team names : 1193

NOCs with >1 Team spelling: 101

=== Top 20 NOCs with most Team name variants ===
  FRA (160): ['Alcyon-6', 'Alcyon-7', 'Allegro', 'Aloha II', 'Amulet-3', 'Amulet-7', 'Ariette-10', 'Ariette-8', 'Astrid III', 'BLO Polo Club, Rugby', 'Baby-1', 'Baby-9', 'Bagatelle Polo Club, Paris', 'C.V.A.-14', 'C.V.A.-7', 'Calimucho', 'Calypse II', 'Camille', 'Carabinier-15', 'Carabinier-5', 'Cercle Nautique de Reims-4', "Cercle de l'Aviron Roubaix-4", 'Chicago Athletic Association', 'Cinara-13', 'Club Nautique de Dieppe-5', 'Club Nautique de Franais-1', 'Club Nautique de Lyon-2', 'Colette-10', 'Colette-12', 'Compigne Polo Club', 'Crabe I-11', 'Crabe I-2', 'Crabe I-3', 'Crabe II-1', 'Crabe II-12', 'Crabe II-4', 'Crocodile-11', 'Crocodile-13', 'Cupidon Viking', 'Damoiselle', 'Demi-Mondaine-15', 'Demi-Mondaine-17', 'Diabolo St Maurice', 'Dick-8', 'Ducky-16', 'Ducky-4', 'EA II', 'Eissero VI', 'Esterel-1', 'Fada', 'Fantlet-2', 'Fantlet-7', 'Favorite-1', 'Fa

In [18]:
# Show ALL NOCs with >1 spelling - full list for the unification table
print('=== ALL NOCs with multiple Team spellings ===')
for _, row in multi_name.iterrows():
    print(f'{row["NOC"]}: {row["Team"]}')

=== ALL NOCs with multiple Team spellings ===
FRA: ['Alcyon-6', 'Alcyon-7', 'Allegro', 'Aloha II', 'Amulet-3', 'Amulet-7', 'Ariette-10', 'Ariette-8', 'Astrid III', 'BLO Polo Club, Rugby', 'Baby-1', 'Baby-9', 'Bagatelle Polo Club, Paris', 'C.V.A.-14', 'C.V.A.-7', 'Calimucho', 'Calypse II', 'Camille', 'Carabinier-15', 'Carabinier-5', 'Cercle Nautique de Reims-4', "Cercle de l'Aviron Roubaix-4", 'Chicago Athletic Association', 'Cinara-13', 'Club Nautique de Dieppe-5', 'Club Nautique de Franais-1', 'Club Nautique de Lyon-2', 'Colette-10', 'Colette-12', 'Compigne Polo Club', 'Crabe I-11', 'Crabe I-2', 'Crabe I-3', 'Crabe II-1', 'Crabe II-12', 'Crabe II-4', 'Crocodile-11', 'Crocodile-13', 'Cupidon Viking', 'Damoiselle', 'Demi-Mondaine-15', 'Demi-Mondaine-17', 'Diabolo St Maurice', 'Dick-8', 'Ducky-16', 'Ducky-4', 'EA II', 'Eissero VI', 'Esterel-1', 'Fada', 'Fantlet-2', 'Fantlet-7', 'Favorite-1', 'Favorite-17', 'Femur-1', 'Femur-18', 'France', 'France-1', 'France-2', 'France-3', 'France-4', '

## 7. Multi-Team Detection (Dash-Suffix Pattern)

**Why**: Team names like "United States-1" or "Germany-2" indicate a country sent multiple teams in the same event. PIPELINE.md Step 1.1.2 says to preserve this as `is_multi_team`. We need to know how common this pattern is and which countries use it.

In [19]:
import re

# Find all Team names matching "-<number>" pattern
dash_suffix = df['Team'].str.extract(r'^(.*)-(\d+)$')
dash_mask = dash_suffix[1].notna()

print(f'Rows with dash-suffix (e.g. "Team-1"): {dash_mask.sum():,} ({dash_mask.sum()/len(df)*100:.2f}%)')
print(f'Unique Team names with dash-suffix: {df.loc[dash_mask, "Team"].nunique()}')
print()

# Which base countries have multi-team entries
multi_team_bases = dash_suffix.loc[dash_mask, 0].unique()
print(f'Unique base country names with multi-team: {len(multi_team_bases)}')
print()
print('Examples:')
display(df.loc[dash_mask, ['Team', 'NOC', 'Year', 'Event']].drop_duplicates().head(20))

Rows with dash-suffix (e.g. "Team-1"): 2,795 (1.11%)
Unique Team names with dash-suffix: 354

Unique base country names with multi-team: 217

Examples:


,Team,NOC,Year,Event
1770,Argentina-2,ARG,2000,Beach Volleyball Men's Beach Volleyball
2013,Argentina-1,ARG,2008,Tennis Men's Doubles
2043,Argentina-2,ARG,2008,Tennis Men's Doubles
2121,Argentina-2,ARG,1924,Tennis Men's Doubles
2175,Argentina-2,ARG,2004,Tennis Men's Doubles
2206,Argentina-1,ARG,2000,Beach Volleyball Men's Beach Volleyball
2358,Argentina-1,ARG,2016,Tennis Men's Doubles
2369,Argentina-2,ARG,2016,Tennis Men's Doubles
2443,Argentina-1,ARG,1924,Tennis Men's Doubles
2486,Argentina-1,ARG,2004,Tennis Men's Doubles


## 8. Garbled / Non-ASCII Team Names

**Why**: PIPELINE.md Section 1.0 mentions "garbled code" handling from the previous attempt. Non-ASCII characters in Team names may indicate encoding issues, Chinese characters, or special symbols (accented letters like e, u, etc. are fine — these are real names). We distinguish: (a) valid accented Latin scripts vs (b) encoding artifacts.

In [20]:
def char_analysis(s):
    """Classify a string by its character composition."""
    if pd.isna(s):
        return 'NaN'
    if '\ufffd' in s or '\x00' in s:
        return 'GARBLED'
    if all(ord(c) < 128 for c in s):
        return 'ASCII'
    return 'NON_ASCII'

df['team_char_type'] = df['Team'].apply(char_analysis)
print('Team name character type distribution:')
print(df['team_char_type'].value_counts())
print()

non_ascii = df[df['team_char_type'] == 'NON_ASCII']['Team'].unique()
print(f'Non-ASCII Team names ({len(non_ascii)} unique):')
for t in sorted(non_ascii)[:30]:
    print(f'  [{t}]')

Team name character type distribution:
team_char_type
ASCII        252377
NON_ASCII       188
Name: count, dtype: int64

Non-ASCII Team names (3 unique):
  [Côte d'Ivoire]
  [São Tomé and Príncipe]
  [Türkiye]


## 9. Exact Duplicate Check

**Why**: PIPELINE.md Step 1.1.1 says duplicates are defined by (Name, NOC, Year, Event). Same athlete, same country, same year, same event appearing twice = recording error. We need to know how many duplicates exist before writing dedup logic.

In [14]:
dup_cols = ['Name', 'NOC', 'Year', 'Event']
dup_mask = df.duplicated(subset=dup_cols, keep=False)
n_dup_rows = dup_mask.sum()

n_dup_groups = df[dup_mask].groupby(dup_cols).ngroups if n_dup_rows > 0 else 0

print(f'Exact duplicate rows (on {dup_cols}): {n_dup_rows:,}')
print(f'Duplicate groups: {n_dup_groups:,}')
print()

if n_dup_rows > 0:
    print('=== Sample of duplicate groups ===')
    display(df[dup_mask].sort_values(dup_cols).head(12))
else:
    print('No exact duplicates found.')

Exact duplicate rows (on ['Name', 'NOC', 'Year', 'Event']): 2,342
Duplicate groups: 773

=== Sample of duplicate groups ===


,Name,Sex,Team,NOC,Year,City,Sport,Event,Medal,team_char_type
76004,A. Dubois,M,Gitana-2,FRA,1900,Paris,Sailing,Sailing Mixed 3-10 Ton,Bronze,ASCII
76005,A. Dubois,M,Gitana-2,FRA,1900,Paris,Sailing,Sailing Mixed 3-10 Ton,Silver,ASCII
95608,A. Rogers,M,Great Britain,GBR,1948,London,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",No medal,ASCII
95609,A. Rogers,M,Great Britain,GBR,1948,London,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",No medal,ASCII
55948,Aase -hoffman-madsen),F,Denmark,DEN,1924,Paris,Art Competitions,Art Competitions Mixed Painting,No medal,ASCII
55949,Aase -hoffman-madsen),F,Denmark,DEN,1924,Paris,Art Competitions,Art Competitions Mixed Painting,No medal,ASCII
7,Abdul Assar,M,Afghanistan,AFG,1948,London,Football,Football Men's Football,No medal,ASCII
8,Abdul Assar,M,Afghanistan,AFG,1948,London,Football,Football Men's Football,No medal,ASCII
177623,Abdul Khan,M,Pakistan,PAK,1948,London,Hockey,Hockey Men's Hockey,No medal,ASCII
177625,Abdul Khan,M,Pakistan,PAK,1948,London,Hockey,Hockey Men's Hockey,No medal,ASCII


## 10. NOC Coverage Check

**Why**: Every NOC in athletes must appear in the medal_counts reference list (Step 1.1.4). NOCs in athletes but NOT in medal_counts are suspicious — possibly typos or historical codes. NOCs in medal_counts but NOT in athletes is also interesting — countries that won medals but have no athlete records?

In [15]:
from src.utils.config import MEDAL_COUNTS_FILE

medals_df = read_csv(MEDAL_COUNTS_FILE)

athlete_nocs = set(df['NOC'].dropna().unique())
medal_nocs = set(medals_df['NOC'].dropna().unique())

print(f'NOCs in athletes    : {len(athlete_nocs)}')
print(f'NOCs in medal_counts: {len(medal_nocs)}')
print()

only_athletes = athlete_nocs - medal_nocs
only_medals = medal_nocs - athlete_nocs
common = athlete_nocs & medal_nocs

print(f'NOCs in BOTH         : {len(common)}')
print(f'NOCs only in athletes: {len(only_athletes)}')
print(f'NOCs only in medals  : {len(only_medals)}')
print()

if only_athletes:
    print(f'NOCs only in athletes (flag for review):')
    for noc in sorted(only_athletes):
        teams = df[df['NOC'] == noc]['Team'].unique()
        print(f'  {noc}: {list(teams)}')

if only_medals:
    print(f'\nNOCs only in medal_counts (check if expected):')
    for noc in sorted(only_medals):
        print(f'  {noc}')

NOCs in athletes    : 234
NOCs in medal_counts: 210

NOCs in BOTH         : 1
NOCs only in athletes: 233
NOCs only in medals  : 209

NOCs only in athletes (flag for review):
  AFG: ['Afghanistan']
  AHO: ['Netherlands Antilles']
  AIN: ['AIN']
  ALB: ['Albania']
  ALG: ['Algeria']
  AND: ['Andorra']
  ANG: ['Angola']
  ANT: ['Antigua and Barbuda']
  ANZ: ['Australasia', 'Sydney Rowing Club']
  ARG: ['Argentina', 'Matrero II', 'Blue Red', 'Argentina-2', 'Pampero', 'Wiking', 'Cupidon III', 'Arcturus', 'Covunco III', 'Tango', 'Argentina-1', 'Acturus', 'Antares', 'Mizar', 'Gullvinge', 'Djinn', 'Rampage', 'Ardilla']
  ARM: ['Armenia']
  ARU: ['Aruba']
  ASA: ['American Samoa']
  AUS: ['Australia', 'Australia-2', 'Australia-1', 'Relampago', 'Buraddoo', 'Gabbiano', 'Vinha', 'Paula', 'Cambria', 'Diablo', 'Naiad', 'Pakaria', 'Moorina', 'Australia/Great Britain', 'Hornet', 'Australia-3', 'Barrenjoey', 'Quando Quando', 'Maryke', 'Falcon VI', 'Amateur Athletic Association', 'Greenoaks Dundee', 'Fa

## 11. Sport & Event Granularity

**Why**: Understanding how many unique Sports and Events exist helps validate against `summerOly_programs.csv`. Also reveals whether Event names are consistent across years.

In [16]:
print(f'Unique Sports : {df["Sport"].nunique()}')
print(f'Unique Events : {df["Event"].nunique()}')
print()

print('=== Top 15 Sports by unique event count ===')
sport_events = df.groupby('Sport')['Event'].apply(lambda x: x.nunique()).sort_values(ascending=False)
display(sport_events.head(15))

Unique Sports : 76
Unique Events : 1041

=== Top 15 Sports by unique event count ===


Sport
Athletics           137
Shooting             99
Swimming             98
Sailing              58
Rowing               51
Wrestling            48
Cycling              44
Weightlifting        41
Boxing               41
Archery              34
Fencing              30
Judo                 30
Art Competitions     29
Canoeing             27
Gymnastics           27
Name: Event, dtype: int64

## 12. City Column Quick Check

**Why**: The City field should match `summerOly_hosts.csv`. Quick check for any Year-City mismatches or unusual values.

In [17]:
print(f'Unique City values: {df["City"].nunique()}')
print()
print('=== City x Year combinations (sample) ===')
city_year = df.groupby('Year')['City'].first().reset_index()
display(city_year)

Unique City values: 23

=== City x Year combinations (sample) ===


,Year,City
0,1896,Athina
1,1900,Paris
2,1904,St. Louis
3,1906,Athina
4,1908,London
...,...,...
26,2008,Beijing
27,2012,London
28,2016,Rio de Janeiro
29,2020,Tokyo


## 13. Summary — Issues Found

After running all checks above, summarize findings here. Each issue should map to a specific step in `clean_athletes.py`:

| # | Issue | Severity | Maps to Step |
|---|-------|----------|--------------|
| 1 | (fill in after running) | | |
| 2 | ... | | |

This summary becomes the implementation checklist for Phase 1.